In [1]:
# ===== Cell 1: Config & imports =====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Root folder where your Stats/ live
PATH = "../../.."   # <-- set this

# Datasets to visualize (no globbing)
DATASETS = ["NCT", "PanNuke", "PANDA", "PatchCamelyon"]  # adjust if needed

# Which metric to visualize from the per-model sheets
# e.g. "test_roc_auc", "test_accuracy", "test_pr_auc"
METRIC = "test_roc_auc"

# Output dir (ALL figures go here)
def out_dir() -> str:
    d = os.path.join(PATH, "Stats", "per_fold_visuals")
    os.makedirs(d, exist_ok=True)
    return d

# Plot look
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ===== Cell 2: Loader + plotting (heatmap w/ numbers & boxplot) =====

def load_dataset_workbook(dataset_name: str):
    """
    Load one dataset workbook produced earlier:
      - SUMMARY (ignored here)
      - one sheet per model containing per-fold metrics.
    Returns dict: {model_name: fold_df}
    """
    xlsx_path = os.path.join(PATH, "Stats", f"{dataset_name}_kfold_summary.xlsx")
    if not os.path.isfile(xlsx_path):
        raise FileNotFoundError(f"Missing workbook for dataset '{dataset_name}': {xlsx_path}")

    xfile = pd.ExcelFile(xlsx_path)
    sheets = {name: xfile.parse(name) for name in xfile.sheet_names}
    sheets.pop("SUMMARY", None)  # we only need per-model fold sheets
    return sheets  # {model_name: fold_df}

def plot_heatmap(ds: str, model_tables: dict, metric: str):
    """
    Heatmap: rows=models, cols=fold ids, values=metric.
    Annotates each cell with its numeric value.
    Saves PNG to Stats/per_fold_visuals/.
    """
    # Collect all fold IDs across models so columns align
    all_folds = sorted({
        int(f) for df in model_tables.values()
        for f in pd.to_numeric(df.get("fold", pd.Series(dtype=float)), errors="coerce").dropna().unique()
    })
    if not all_folds:
        print(f"[{ds}] No fold indices found; skipping heatmap.")
        return

    models = sorted(model_tables.keys())
    mat = np.full((len(models), len(all_folds)), np.nan, dtype=float)

    # Fill matrix
    for i, m in enumerate(models):
        df = model_tables[m]
        if metric not in df.columns or "fold" not in df.columns:
            continue
        fold = pd.to_numeric(df["fold"], errors="coerce")
        vals = pd.to_numeric(df[metric], errors="coerce")
        keep = fold.notna() & vals.notna()
        for f, v in zip(fold[keep].astype(int), vals[keep].astype(float)):
            if f in all_folds:
                j = all_folds.index(f)
                mat[i, j] = v

    # If everything is NaN, bail
    if np.isnan(mat).all():
        print(f"[{ds}] No valid '{metric}' values; skipping heatmap.")
        return

    # Color scale (ignore NaNs)
    vmin = float(np.nanmin(mat))
    vmax = float(np.nanmax(mat))

    # Plot
    fig, ax = plt.subplots(figsize=(max(8, len(all_folds)*0.8), max(6, len(models)*0.45)))
    im = ax.imshow(mat, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax, cmap="viridis")
    ax.set_xticks(range(len(all_folds))); ax.set_xticklabels(all_folds)
    ax.set_yticks(range(len(models)));    ax.set_yticklabels(models)
    ax.set_xlabel("Fold")
    ax.set_ylabel("Model")
    ax.set_title(f"{ds} — {metric} per fold")

    # Add numeric annotations (3 d.p.), skip NaNs
    for i in range(len(models)):
        for j in range(len(all_folds)):
            val = mat[i, j]
            if np.isnan(val):
                continue
            # Choose text color based on normalized color (brighter cell -> use black text)
            normed = im.norm(val)
            text_color = "black" if normed > 0.6 else "white"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, color=text_color)

    # Colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(metric)
    fig.tight_layout()

    out_path = os.path.join(out_dir(), f"{ds}__{metric}__heatmap.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[{ds}] Saved heatmap → {out_path}")

def plot_boxplot(ds: str, model_tables: dict, metric: str):
    """
    Boxplot: distribution of metric across folds per model.
    Saves PNG to Stats/per_fold_visuals/.
    """
    models, data = [], []
    for m, df in model_tables.items():
        if metric not in df.columns:
            continue
        vals = pd.to_numeric(df[metric], errors="coerce").dropna().values
        if len(vals) > 0:
            models.append(m)
            data.append(vals)

    if not data:
        print(f"[{ds}] No valid '{metric}' values; skipping boxplot.")
        return

    fig, ax = plt.subplots(figsize=(max(8, len(models)*0.7), 6))
    ax.boxplot(data, labels=models, vert=True, showmeans=True)
    ax.set_ylabel(metric)
    ax.set_title(f"{ds} — {metric} per fold (boxplot)")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()

    out_path = os.path.join(out_dir(), f"{ds}__{metric}__boxplot.png")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[{ds}] Saved boxplot → {out_path}")


In [3]:
# ===== Cell 3: Generate figures only =====
for ds in DATASETS:
    try:
        model_tables = load_dataset_workbook(ds)  # {model_name: fold_df}
    except Exception as e:
        print(f"[{ds}] Skipping (load error): {e}")
        continue

    plot_heatmap(ds, model_tables, METRIC)
    plot_boxplot(ds, model_tables, METRIC)

print("Done. Created heatmaps and boxplots only, saved under Stats/per_fold_visuals/.")

[NCT] Saved heatmap → Stats/per_fold_visuals/NCT__test_roc_auc__heatmap.png
[NCT] Saved boxplot → Stats/per_fold_visuals/NCT__test_roc_auc__boxplot.png
[PanNuke] Saved heatmap → Stats/per_fold_visuals/PanNuke__test_roc_auc__heatmap.png
[PanNuke] Saved boxplot → Stats/per_fold_visuals/PanNuke__test_roc_auc__boxplot.png
[PANDA] Saved heatmap → Stats/per_fold_visuals/PANDA__test_roc_auc__heatmap.png
[PANDA] Saved boxplot → Stats/per_fold_visuals/PANDA__test_roc_auc__boxplot.png
[PatchCamelyon] Saved heatmap → Stats/per_fold_visuals/PatchCamelyon__test_roc_auc__heatmap.png
[PatchCamelyon] Saved boxplot → Stats/per_fold_visuals/PatchCamelyon__test_roc_auc__boxplot.png
Done. Created heatmaps and boxplots only, saved under Stats/per_fold_visuals/.
